<a href="https://colab.research.google.com/github/EllenKAlves/AuroraSinger_FIAP/blob/main/MGPEB.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MGPEB — Módulo de Gerenciamento de Pouso e Estabilização de Base
# Missão: Aurora Siger | Destino: Marte

ETAPA 1 — Definição e Cadastro dos Módulos de Pouso

In [62]:
from collections import deque

In [63]:
# Cada módulo é um dicionário com os atributos básicos da missão. Estrutura: nome, tipo, prioridade (1=mais urgente), combustivel (%),massa (t), criticidade, horario_chegada_orbita

modulos_definidos = [
    {
        "nome": "Habitat Alpha",
        "tipo": "Habitação",
        "prioridade": 1,
        "combustivel": 85,
        "massa": 12.0,
        "criticidade": "Alta",
        "horario_chegada_orbita": "08:00"
    },
    {
        "nome": "Reator Solar",
        "tipo": "Energia",
        "prioridade": 2,
        "combustivel": 60,
        "massa": 8.5,
        "criticidade": "Alta",
        "horario_chegada_orbita": "09:30"
    },
    {
        "nome": "MedBase",
        "tipo": "Suporte Médico",
        "prioridade": 2,
        "combustivel": 90,
        "massa": 5.5,
        "criticidade": "Alta",
        "horario_chegada_orbita": "10:00"
    },
    {
        "nome": "BioLab",
        "tipo": "Laboratório Científico",
        "prioridade": 3,
        "combustivel": 72,
        "massa": 6.0,
        "criticidade": "Média",
        "horario_chegada_orbita": "11:00"
    },
    {
        "nome": "LogiStore",
        "tipo": "Logística",
        "prioridade": 4,
        "combustivel": 45,
        "massa": 15.0,
        "criticidade": "Baixa",
        "horario_chegada_orbita": "13:00"
    },
    {
        "nome": "TerraBot",
        "tipo": "Mineração",
        "prioridade": 5,
        "combustivel": 30,
        "massa": 18.0,
        "criticidade": "Baixa",
        "horario_chegada_orbita": "15:00"
    },
]

In [64]:
# Fila principal: módulos aguardando autorização de pouso (ordem de chegada)
fila_pouso = deque()

# Lista de módulos já pousados com sucesso
lista_pousados = []

# Lista de módulos em espera (pouso adiado)
lista_em_espera = []

# Pilha (stack) de módulos em alerta (LIFO): o último módulo a entrar em alerta
# é o primeiro a ser tratado pelo sistema de recuperação.
pilha_em_alerta = []

# Pilha (stack) de histórico de pousos: o último pouso bem-sucedido fica no topo (LIFO). Útil para auditoria rápida e rollback.
# Operações usadas: append() empilha no topo | pop() ou [-1] acessa o topo.
pilha_historico_pousos = []

In [65]:
# Cadastra todos os módulos na fila principal
for modulo in modulos_definidos:
    fila_pouso.append(modulo)

print("   🛸 MGPEB — Missão Aurora Siger inicializado")
print(f"\n📋 {len(fila_pouso)} módulos cadastrados na fila de pouso:\n")
for i, m in enumerate(fila_pouso, 1):
    print(f"  {i}. {m['nome']} | Tipo: {m['tipo']} | Prioridade: {m['prioridade']} | "
          f"Combustível: {m['combustivel']}% | Massa: {m['massa']}t | "
          f"Criticidade: {m['criticidade']} | Chegada: {m['horario_chegada_orbita']}")
print()

   🛸 MGPEB — Missão Aurora Siger inicializado

📋 6 módulos cadastrados na fila de pouso:

  1. Habitat Alpha | Tipo: Habitação | Prioridade: 1 | Combustível: 85% | Massa: 12.0t | Criticidade: Alta | Chegada: 08:00
  2. Reator Solar | Tipo: Energia | Prioridade: 2 | Combustível: 60% | Massa: 8.5t | Criticidade: Alta | Chegada: 09:30
  3. MedBase | Tipo: Suporte Médico | Prioridade: 2 | Combustível: 90% | Massa: 5.5t | Criticidade: Alta | Chegada: 10:00
  4. BioLab | Tipo: Laboratório Científico | Prioridade: 3 | Combustível: 72% | Massa: 6.0t | Criticidade: Média | Chegada: 11:00
  5. LogiStore | Tipo: Logística | Prioridade: 4 | Combustível: 45% | Massa: 15.0t | Criticidade: Baixa | Chegada: 13:00
  6. TerraBot | Tipo: Mineração | Prioridade: 5 | Combustível: 30% | Massa: 18.0t | Criticidade: Baixa | Chegada: 15:00



# ETAPA 2 — Regras de Decisão com Portas Lógicas



In [66]:
# Condições simuladas do ambiente marciano e da área de pouso
condicoes_ambiente = {
    "area_de_pouso_livre": True,       # Área disponível para receber o módulo
    "tempestade_de_areia": False,      # Tempestade ativa bloqueia qualquer pouso
    "sensores_operacionais": True,     # Sensores de navegação funcionando
    "comunicacao_terra": True,         # Link com controle na Terra ativo
    "reserva_emergencial": True       # Reserva de combustível para manobras críticas
}

In [67]:
# Função que avalia se o pouso de um módulo pode ser autorizado.

    #Regras booleanas aplicadas:
    # Combustível mínimo de 60% para garantir manobra de pouso segura
    # Área de pouso deve estar livre
    # Não pode haver tempestade de areia ativa
    # Sensores de navegação precisam estar operacionais
    # Módulos críticos exigem também comunicação ativa com a Terra

def autorizar_pouso(modulo, ambiente, emergencia_medica=False):
    motivos = []

    # Mapeamento das variáveis (baseado na sua lógica do relatório)
    # C = Combustível, A = Atmosfera, P = Plataforma, S = Sensores
    C = modulo["combustivel"] >= 60
    A = not ambiente["tempestade_de_areia"]
    P = ambiente["area_de_pouso_livre"]
    S = ambiente["sensores_operacionais"]

    # Tratamento especial para MED-05 (Emergência Médica)
    # Regra: Se MED-05 e emergência, o pouso é liberado mesmo sem Plataforma (P)
    if "id" in modulo and modulo["id"] == "MED-05" and emergencia_medica:
        autorizado = C and A and S and (P or True)
    else:
        # Regra padrão: AUTORIZADO = C AND A AND P AND S
        autorizado = C and A and P and S

    # Geração dos motivos de bloqueio (apenas para logs/feedback)
    if not C: motivos.append("Combustível insuficiente (< 60%)")
    if not A: motivos.append("Condições atmosféricas críticas (tempestade)")
    # Check P only if it's not an emergency MED-05 situation
    if not P and not ("id" in modulo and modulo["id"] == "MED-05" and emergencia_medica):
        motivos.append("Plataforma ocupada")
    if not S: motivos.append("Falha nos sensores")

    return autorizado, motivos

In [68]:
## Teste da função que avalia o pouso
modulo_teste = modulos_definidos[5]
teste = autorizar_pouso(modulo_teste, condicoes_ambiente)

print(f"Módulo: {modulo_teste['nome']}")
print(f"Autorizado: {teste[0]}")
print(f"Motivos de bloqueio: {teste[1]}")

Módulo: TerraBot
Autorizado: False
Motivos de bloqueio: ['Combustível insuficiente (< 60%)']


# ETAPA 3 — Algorítmos de Busca, Ordenação e Simulação de Pouso


In [69]:
# Algoritmos de busca
#Busca linear: encontra o módulo com menor nível de combustível.
def buscar_menor_combustivel(lista):
    if not lista:
        return None
    menor = lista[0]
    for m in lista:
        if m["combustivel"] < menor["combustivel"]:
            menor = m
    return menor

In [70]:
# Teste
resultado = buscar_menor_combustivel(modulos_definidos)
print(f"Módulo com menor combustível: {resultado['nome']} ({resultado['combustivel']}%)")

Módulo com menor combustível: TerraBot (30%)


In [71]:
#Busca linear: encontra o módulo com maior prioridade (menor número = mais urgente)
def buscar_maior_prioridade(lista):
    if not lista:
        return None
    mais_urgente = lista[0]
    for m in lista:
        if m["prioridade"] < mais_urgente["prioridade"]:
            mais_urgente = m
    return mais_urgente

In [72]:
# Teste
resultado1 = buscar_maior_prioridade(modulos_definidos)
print(f"Módulo com maior prioridade: {resultado1['nome']} ({resultado1['prioridade']})")

Módulo com maior prioridade: Habitat Alpha (1)


In [73]:
# Busca linear: retorna todos os módulos de um determinado tipo
def buscar_por_tipo(lista, tipo_buscado):
    return [m for m in lista if m["tipo"].lower() == tipo_buscado.lower()]

In [74]:
# Teste
resultado2 = buscar_por_tipo(modulos_definidos, "Energia")
for m in resultado2:
    print(f"• {m['nome']}")

• Reator Solar


In [75]:
# Algoritmo de ordenação (Bubble Sort)
## Ordena a lista de módulos por prioridade usando Bubble Sort
## Prioridade 1 = mais urgente → deve ficar no início da fila
def ordenar_por_prioridade(lista):

    lista_ord = list(lista)  # cria uma cópia para não alterar o original
    n = len(lista_ord)
    for i in range(n):
        for j in range(0, n - i - 1):
            if lista_ord[j]["prioridade"] > lista_ord[j + 1]["prioridade"]:
                lista_ord[j], lista_ord[j + 1] = lista_ord[j + 1], lista_ord[j]
    return lista_ord

In [76]:
teste_lista_ord = ordenar_por_prioridade(modulos_definidos)

for m in teste_lista_ord:
    print(f"Prioridade {m['prioridade']} — {m['nome']}")

Prioridade 1 — Habitat Alpha
Prioridade 2 — Reator Solar
Prioridade 2 — MedBase
Prioridade 3 — BioLab
Prioridade 4 — LogiStore
Prioridade 5 — TerraBot


In [77]:
# Simulação principal de autorização de pouso
print("=" * 55)
print("   ☢ Iniciando Simulação de Autorização de Pouso")
print("=" * 55)
print()

# Função que avalia se o pouso de um módulo pode ser autorizado.

    #Regras booleanas aplicadas:
    # Combustível mínimo de 60% para garantir manobra de pouso segura
    # Área de pouso deve estar livre
    # Não pode haver tempestade de areia ativa
    # Sensores de navegação precisam estar operacionais
    # Módulos críticos exigem também comunicação ativa com a Terra

def autorizar_pouso(modulo, ambiente, emergencia_medica=False):
    motivos = []

    # Mapeamento das variáveis (baseado na sua lógica do relatório)
    # C = Combustível, A = Atmosfera, P = Plataforma, S = Sensores
    C = modulo["combustivel"] >= 60
    A = not ambiente["tempestade_de_areia"]
    P = ambiente["area_de_pouso_livre"]
    S = ambiente["sensores_operacionais"]

    # Tratamento especial para MED-05 (Emergência Médica)
    # Regra: Se MED-05 e emergência, o pouso é liberado mesmo sem Plataforma (P)
    if "id" in modulo and modulo["id"] == "MED-05" and emergencia_medica:
        autorizado = C and A and S and (P or True)
    else:
        # Regra padrão: AUTORIZADO = C AND A AND P AND S
        autorizado = C and A and P and S

    # Geração dos motivos de bloqueio (apenas para logs/feedback)
    if not C: motivos.append("Combustível insuficiente (< 60%)")
    if not A: motivos.append("Condições atmosféricas críticas (tempestade)")
    # Check P only if it's not an emergency MED-05 situation
    if not P and not ("id" in modulo and modulo["id"] == "MED-05" and emergencia_medica):
        motivos.append("Plataforma ocupada")
    if not S: motivos.append("Falha nos sensores")

    return autorizado, motivos

# Algoritmo de ordenação (Bubble Sort)
## Ordena a lista de módulos por prioridade usando Bubble Sort
## Prioridade 1 = mais urgente → deve ficar no início da fila
def ordenar_por_prioridade(lista):

    lista_ord = list(lista)  # cria uma cópia para não alterar o original
    n = len(lista_ord)
    for i in range(n):
        for j in range(0, n - i - 1):
            if lista_ord[j]["prioridade"] > lista_ord[j + 1]["prioridade"]:
                lista_ord[j], lista_ord[j + 1] = lista_ord[j + 1], lista_ord[j]
    return lista_ord

# Reorganiza a fila por prioridade antes de processar
fila_ordenada = ordenar_por_prioridade(list(fila_pouso))
fila_pouso = deque(fila_ordenada)

print("Condições do ambiente marciano:")
for cond, valor in condicoes_ambiente.items():
    status = "✅ OK" if valor else "❌ FALHA"
    # NOT é aplicado à tempestade: False = sem tempestade = OK
    if cond == "tempestade_de_areia":
        status = "✅ OK" if not valor else "❌ TEMPESTADE ATIVA"
    print(f"   {cond.replace('_', ' ').title()}: {status}")
print()

# Processa cada módulo da fila
while fila_pouso:
    modulo = fila_pouso.popleft()  # retira o primeiro da fila (FIFO)
    autorizado, motivos = autorizar_pouso(modulo, condicoes_ambiente)

    # Todos os prints e condições abaixo devem estar dentro do loop (identados)
    print(f"☢ Módulo: {modulo['nome']} | Tipo: {modulo['tipo']} | "
          f"Combustível: {modulo['combustivel']}% | Criticidade: {modulo['criticidade']}")

    # IF / ELIF / ELSE: roteamento do módulo após avaliação
    if autorizado:
        print(f"   ✅ POUSO AUTORIZADO")
        lista_pousados.append(modulo)
        pilha_historico_pousos.append(modulo)       # empilha no topo (LIFO)
    elif modulo["criticidade"] == "Alta":
        print(f"   ❌ POUSO BLOQUEADO — módulo crítico enviado ao ALERTA")
        for motivo in motivos:
            print(f"      → {motivo}")
        # IMPLEMENTAÇÃO LIFO: Adiciona no topo da pilha
        pilha_em_alerta.append(modulo)
    else:
        print(f"   ❌ POUSO BLOQUEADO — módulo enviado à ESPERA")
        for motivo in motivos:
            print(f"      → {motivo}")
        lista_em_espera.append(modulo)

    print("-" * 55)

# --- O loop WHILE terminou aqui, agora processamos os alertas pendentes ---

# Processamento LIFO: Resolvendo os alertas que ficaram na pilha
if len(pilha_em_alerta) > 0:
    print("\n⚠️ Processando alertas (LIFO)...")
    while pilha_em_alerta:
        modulo_alerta = pilha_em_alerta.pop() # Remove o último que entrou
        print(f"   → Resolvendo alerta do módulo: {modulo_alerta['nome']}")


   ☢ Iniciando Simulação de Autorização de Pouso

Condições do ambiente marciano:
   Area De Pouso Livre: ✅ OK
   Tempestade De Areia: ✅ OK
   Sensores Operacionais: ✅ OK
   Comunicacao Terra: ✅ OK
   Reserva Emergencial: ✅ OK

☢ Módulo: Habitat Alpha | Tipo: Habitação | Combustível: 85% | Criticidade: Alta
   ✅ POUSO AUTORIZADO
-------------------------------------------------------
☢ Módulo: Reator Solar | Tipo: Energia | Combustível: 60% | Criticidade: Alta
   ✅ POUSO AUTORIZADO
-------------------------------------------------------
☢ Módulo: MedBase | Tipo: Suporte Médico | Combustível: 90% | Criticidade: Alta
   ✅ POUSO AUTORIZADO
-------------------------------------------------------
☢ Módulo: BioLab | Tipo: Laboratório Científico | Combustível: 72% | Criticidade: Média
   ✅ POUSO AUTORIZADO
-------------------------------------------------------
☢ Módulo: LogiStore | Tipo: Logística | Combustível: 45% | Criticidade: Baixa
   ❌ POUSO BLOQUEADO — módulo enviado à ESPERA
      → 

In [79]:
# Relatório final

print()
print("=" * 55)
print("   📊 RELATÓRIO FINAL — MGPEB Aurora Siger")
print("=" * 55)

print(f"\n✅ Módulos pousados com sucesso ({len(lista_pousados)}):")
for m in lista_pousados:
    print(f"   • {m['nome']} ({m['tipo']})")

print(f"\n⏳ Módulos em espera ({len(lista_em_espera)}):")
for m in lista_em_espera:
    print(f"   • {m['nome']} ({m['tipo']})")

print(f"\n⚠️  Módulos em alerta crítico ({len(pilha_em_alerta)}):")
for m in pilha_em_alerta:
    print(f"   • {m['nome']} ({m['tipo']})")

print(f"\n📚 Histórico de pousos (pilha LIFO — topo = último a pousar):")
# Percorre do topo para a base: o último empilhado aparece primeiro
for m in reversed(pilha_historico_pousos):
    print(f"   • {m['nome']} ({m['tipo']})")


   📊 RELATÓRIO FINAL — MGPEB Aurora Siger

✅ Módulos pousados com sucesso (4):
   • Habitat Alpha (Habitação)
   • Reator Solar (Energia)
   • MedBase (Suporte Médico)
   • BioLab (Laboratório Científico)

⏳ Módulos em espera (2):
   • LogiStore (Logística)
   • TerraBot (Mineração)

⚠️  Módulos em alerta crítico (0):

📚 Histórico de pousos (pilha LIFO — topo = último a pousar):
   • BioLab (Laboratório Científico)
   • MedBase (Suporte Médico)
   • Reator Solar (Energia)
   • Habitat Alpha (Habitação)
